# Questão 7 — Previsão de Demanda

## Objetivo

Construir um modelo baseline simples para prever a demanda diária do produto **Motor de Popa Yamaha Evo Dash 155HP** durante o mês de Janeiro de 2024.

## Contexto de negócio

A empresa precisa reduzir decisões baseadas em feeling e passar a apoiar o planejamento de compras com uma previsão objetiva de demanda.

## Abordagem

Será utilizado um baseline de **média móvel de 7 dias**, calculado apenas com dados históricos disponíveis até a data da previsão, evitando vazamento de informação.

## Fonte de dados

A análise será feita sobre a camada gold do projeto, utilizando:

- `lh_nautical.04_gold.fct_vendas`
- `lh_nautical.04_gold.dim_produto`

Essa abordagem mantém aderência ao modelo dimensional já construído no projeto.

In [0]:
# Importação das bibliotecas necessárias para manipulação de dados
import pandas as pd
import numpy as np

# Leitura das tabelas
df_vendas = spark.table("lh_nautical.04_gold.fct_vendas")
df_produtos = spark.table("lh_nautical.04_gold.dim_produto")

# Conversão para pandas para facilitar a construção do baseline temporal
pdf_vendas = df_vendas.toPandas()
pdf_produtos = df_produtos.toPandas()

# Garantia do tipo correto da coluna de data
pdf_vendas["sale_date"] = pd.to_datetime(pdf_vendas["sale_date"])

In [0]:
# Junção entre fato de vendas e dimensão de produto
# Necessária porque o nome do produto está na dim_produto, não na fato
df_base = pdf_vendas.merge(
    pdf_produtos[["product_id", "product_name"]],
    on="product_id",
    how="left"
)

# Conferência inicial da base analítica
df_base.head()

In [0]:
# Definição do produto alvo
produto_alvo = "Motor de Popa Yamaha Evo Dash 155HP"

# Filtra apenas as vendas do produto alvo
df_produto = df_base[df_base["product_name"] == produto_alvo].copy()

# Validação básica do filtro
print("Quantidade de registros do produto:", len(df_produto))
df_produto.head()

In [0]:
# Agregação das vendas em base diária
# A métrica prevista será quantidade vendida por dia
df_diario = (
    df_produto
    .groupby("sale_date", as_index=False)["quantity"]
    .sum()
    .sort_values("sale_date")
)

df_diario.head()

## Construção da série temporal completa

Mesmo para um único produto, dias sem venda precisam existir explicitamente na série.
Sem isso, a média móvel seria calculada apenas sobre dias com transação registrada, distorcendo a previsão.

Por esse motivo, será criado um calendário diário completo entre a menor e a maior data da série do produto, preenchendo dias sem venda com zero.

In [0]:
# Criação de calendário completo entre a menor e a maior data observada
datas_completas = pd.date_range(
    start=df_diario["sale_date"].min(),
    end=df_diario["sale_date"].max(),
    freq="D"
)

df_calendario = pd.DataFrame({"sale_date": datas_completas})

# Left join entre o calendário e a série agregada do produto
df_serie = df_calendario.merge(df_diario, on="sale_date", how="left")

# Tratamento dos dias sem venda
df_serie["quantity"] = df_serie["quantity"].fillna(0)

# Ordenação final da série temporal
df_serie = df_serie.sort_values("sale_date").reset_index(drop=True)

df_serie.head(10)

In [0]:
# Separação dos períodos conforme a premissa da questão
# Treino: até 31/12/2023
# Teste: todo o mês de Janeiro de 2024

train = df_serie[df_serie["sale_date"] <= "2023-12-31"].copy()
test = df_serie[
    (df_serie["sale_date"] >= "2024-01-01") &
    (df_serie["sale_date"] <= "2024-01-31")
].copy()

print("Quantidade de dias no treino:", len(train))
print("Quantidade de dias no teste:", len(test))

## Construção do baseline

O baseline será calculado com média móvel de 7 dias.

Para cada data, a previsão será a média das quantidades vendidas nos 7 dias anteriores.  
Para impedir data leakage, a janela será deslocada com `shift(1)`, garantindo que o valor real do próprio dia não entre no cálculo da previsão.

In [0]:
# Construção da previsão baseline com média móvel de 7 dias
# O uso de shift(1) garante que a previsão utilize apenas informações passadas
df_serie["forecast"] = (
    df_serie["quantity"]
    .shift(1)
    .rolling(window=7)
    .mean()
)

# Seleção do período de teste com as previsões calculadas
test = df_serie[
    (df_serie["sale_date"] >= "2024-01-01") &
    (df_serie["sale_date"] <= "2024-01-31")
].copy()

test.head(10)

In [0]:
# Cálculo do erro absoluto médio (MAE)
# A métrica compara a previsão diária com a quantidade real observada
mae = np.mean(np.abs(test["quantity"] - test["forecast"]))

print(f"MAE do baseline: {mae:.2f}")

In [0]:
# Visualização detalhada do período de teste
# Essa tabela ajuda na validação do comportamento do baseline
test[["sale_date", "quantity", "forecast"]].head(15)

In [0]:
# Filtra a primeira semana de Janeiro de 2024
primeira_semana = test[
    (test["sale_date"] >= "2024-01-01") &
    (test["sale_date"] <= "2024-01-07")
].copy()

# Soma das previsões do período
soma_previsao_primeira_semana = primeira_semana["forecast"].sum()

# Arredondamento para número inteiro, conforme solicitado na questão
resultado_q72 = int(round(soma_previsao_primeira_semana, 0))

print("Soma total da previsão de 01/01 a 07/01:", resultado_q72)

In [0]:
# Detalhamento das previsões da primeira semana de Janeiro
primeira_semana[["sale_date", "quantity", "forecast"]]

## Questão 7

O baseline não se mostrou adequado para esse produto específico.

Isso ocorre porque o produto apresenta um padrão de vendas intermitente, com longos períodos sem vendas. Como a média móvel de 7 dias depende exclusivamente do histórico recente, e os últimos dias do período de treino possuem valor zero, o modelo passa a prever zero para todo o período de teste.

Esse comportamento evidencia uma limitação importante do método para séries com baixa frequência de vendas.


## Questão 7.2 — Validação

A soma total da previsão de vendas para o produto **Motor de Popa Yamaha Evo Dash 155HP** durante a primeira semana de Janeiro de 2024 foi **0 unidades**.

## Questão 7.3 — Explicação

### Como o baseline foi construído?

O baseline foi construído a partir da quantidade vendida por dia do produto analisado. Primeiro, as vendas foram agregadas em nível diário. Em seguida, foi criado um calendário completo para garantir que os dias sem venda também estivessem presentes na série com valor zero. A previsão de cada dia foi então calculada como a média das vendas dos 7 dias anteriores.

### Como evitou data leakage?

O vazamento de dados foi evitado com o uso de `shift(1)` antes da média móvel. Isso garante que a previsão de uma data utilize apenas valores históricos disponíveis até o dia anterior, sem acessar o valor real do próprio dia previsto nem qualquer informação futura.

### Uma limitação do modelo proposto

Uma limitação relevante do modelo é sua incapacidade de lidar com séries temporais esparsas (intermittent demand). Como o método depende exclusivamente do histórico recente, períodos sem venda fazem com que a previsão colapse para zero, mesmo que o produto volte a vender posteriormente.

In [0]:
# Gráfico simples comparando valores reais e previstos no período de teste
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
plt.plot(test["sale_date"], test["quantity"], label="Real")
plt.plot(test["sale_date"], test["forecast"], label="Previsto")
plt.title("Real x Previsto — Janeiro de 2024")
plt.xlabel("Data")
plt.ylabel("Quantidade vendida")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [0]:
%sql
SHOW TABLES IN lh_nautical.04_gold;

In [0]:
# Últimos 15 dias do treino
train.tail(15)

In [0]:
# Ver os últimos dias antes de 2024
df_serie[df_serie["sale_date"] >= "2023-12-20"]